# Exp 4B — Long-context baseline (V2, semantic edition)
**Question:** if the complete policy corpus fits in context, is RAG necessary at all? RAG must later beat this on at least one of: accuracy, false approvals, policy-version precision, token cost, latency, scalability, evidence traceability.

**Input:** the claim (bill, dates, project, employee note) plus the policy corpus in the system prompt. No enterprise tools or records. The prompt asks the model to apply the version in force on the transaction date and to let regional addenda override global rules, and adds one sentence that claim dates are real (they may be later than the model's training data).
**Variants (Exp 4A sizing):** *full* = all 22 documents verbatim (about 49-52k tokens); *lean* = normative clause text only (about 17-19k tokens).
**Models (temperature 0):** `openai/gpt-4o-mini`, `google/gemini-2.5-flash-lite`; `llama3.2:3b` for lean only, with `num_ctx` 24,576 (full is infeasible locally, see 4A).
**Data:** 70 development claims. **Reference points:** Exp 3 (no policy) and the Exp 2 rule baselines.

In [1]:
import json, os, re, sys
from pathlib import Path
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
ROOT = Path.cwd().parents[1]; sys.path.insert(0, str(ROOT))
from src import config as C, llm, llm_exp, evaluate, metrics_ext
C.load_env()
pd.set_option('display.width', 250, 'display.max_columns', 40, 'display.max_colwidth', 110, 'display.max_rows', 200)
docs = sorted((C.POLICY/'source_documents').glob('*.md'))
def strip_commentary(text):
    out, lines, i = [], text.split('\n'), 0
    while i < len(lines):
        if lines[i].startswith('Interpretive guidance:'):
            i += 1
            while i < len(lines) and not lines[i].strip(): i += 1
            i += 1; continue
        out.append(lines[i]); i += 1
    return re.sub(r'\n{3,}', '\n\n', '\n'.join(out))
FULL = '\n\n=====\n\n'.join(p.read_text(encoding='utf-8') for p in docs); LEAN = '\n\n=====\n\n'.join(strip_commentary(p.read_text(encoding='utf-8')) for p in docs)
SYS = ("You are ExpenseGuard, a finance assistant deciding whether an employee expense claim is ready for reimbursement, using ONLY the company policy corpus below and the claim. "
       "Policies have effective dates and regions: apply the version in force on the transaction date and let regional addenda override global rules; a Finance circular amends the clause it names from its effective date. "
       "Convert currencies only if a conversion is needed and state the assumption. Claim dates are real dates and may be later than your training data. If a needed record (approval, travel request, conference registration) is not part of the claim, "
       "you cannot see it: say what is missing rather than assuming it exists.\n" + llm_exp.SCHEMA + "\n\nCOMPANY POLICY CORPUS:\n")
SYSTEMS = {'full': SYS + FULL, 'lean': SYS + LEAN}
user = lambda c: "CLAIM:\n" + json.dumps(llm_exp.visible(c), indent=1)
cases = llm_exp.cases_for('DEVELOPMENT')
PAID, LOCAL, GEM = os.environ['PAID_MODEL'], os.environ['LOCAL_MODEL'], 'google/gemini-2.5-flash-lite'
PLAN = [('full', PAID), ('full', GEM), ('lean', PAID), ('lean', GEM), ('lean', LOCAL)]
PRICE = {PAID: (0.15, 0.60), GEM: (0.10, 0.40), LOCAL: (0, 0)}
tok = {'full': {PAID: 49095, GEM: 51870}, 'lean': {PAID: 16592, GEM: 18893}}    # measured in Exp 4A
proj = sum((tok[v][m] + 350) * PRICE[m][0] * 70 / 1e6 + 90 * PRICE[m][1] * 70 / 1e6 for v, m in PLAN if m != LOCAL)
print('Projected paid cost: $%.3f | spent so far $%.4f of $%s' % (proj, llm.spent(), os.environ['MAX_BUDGET_USD'])); assert proj < 1.5, 'above the approved cap'
CONFIG = dict(split='DEVELOPMENT', temperature=0, policy='corpus in system prompt', tools='none', max_tokens=600)

Projected paid cost: $1.210 | spent so far $0.5778 of $3.50


## Run (live calls, cached by content). Local model uses a 24,576-token window

In [2]:
res = {}
for variant, m in PLAN:
    llm.NUM_CTX = 24576 if m == LOCAL else 16384
    res[(variant, m)] = llm_exp.run('EXP04B_' + variant.upper(), m, 'DEVELOPMENT', SYSTEMS[variant], user, cases=cases, config={**CONFIG, 'variant': variant, 'num_ctx': llm.NUM_CTX})
    s = res[(variant, m)][0]; print(variant, m, s['correct'], '/', s['n'], '| in_tok', s['input_tokens'], '| cost $%.4f' % s['total_cost_usd'])
llm.NUM_CTX = 16384
print('spent so far $%.4f' % llm.spent())

full openai/gpt-4o-mini 22 / 70 | in_tok 3447753 | cost $0.2707


full google/gemini-2.5-flash-lite 23 / 70 | in_tok 3641881 | cost $0.2949


lean openai/gpt-4o-mini 20 / 70 | in_tok 1172543 | cost $0.0946


lean google/gemini-2.5-flash-lite 25 / 70 | in_tok 1333491 | cost $0.1125


lean llama3.2:3b 14 / 70 | in_tok 1184037 | cost $0.0000
spent so far $1.3505


## Comparison table: standard metrics (Exp 3 and Exp 2 references below)

In [3]:
EX2 = C.RESULTS/'development'/'exp02_rules'
base = {k: json.loads((EX2/f'summary_{t}_development.json').read_text()) for k, t in {'rules 2a':'2a_asis','rules 2b':'2b_frozen','rules 2c':'2c_regex'}.items()}
e3 = {k: json.loads(p.read_text()) for k, p in {'gpt-4o-mini':C.RESULTS/'development'/'exp03_no_policy'/'summary_openai_gpt-4o-mini.json','gemini-lite':C.RESULTS/'development'/'exp03_no_policy'/'summary_google_gemini-2.5-flash-lite.json','llama3.2:3b':C.RESULTS/'development'/'exp03_no_policy'/'summary_llama3.2_3b.json'}.items()}
def row(name, s): return {'system':name, 'correct/N':f"{s['correct']}/{s['n']}", 'CDR':s['correct_disposition_rate'], 'Wilson95':tuple(s['wilson95']), 'false_approvals':f"{s['false_approvals']}/{s['non_approvable']}", 'FAR':s['false_approval_rate'],
        'human_review_rate':s['human_review_rate'], 'schema_valid':f"{s.get('schema_valid', s['n'])}/{s['n']}", 'errors':s['errors'], 'median_ms':s['median_latency_ms'], 'p95_ms':s['p95_latency_ms'], 'in_tok':s['input_tokens'], 'out_tok':s['output_tokens'], 'cost_usd':s['total_cost_usd']}
label = lambda v, m: f"4B-{v} {m.split('/')[-1]}"
tab = pd.DataFrame([row(label(v, m), res[(v, m)][0]) for v, m in PLAN] + [row('Exp3 ' + k, v) for k, v in e3.items()] + [row(k, v) for k, v in base.items()]); tab

,system,correct/N,CDR,Wilson95,false_approvals,FAR,human_review_rate,schema_valid,errors,median_ms,p95_ms,in_tok,out_tok,cost_usd
0,4B-full gpt-4o-mini,22/70,0.3143,"(0.2176, 0.4303)",0/52,0.0000,0.0000,70/70,0,2827.5000,4185.000,3447753,4874,0.270718
1,4B-full gemini-2.5-flash-lite,23/70,0.3286,"(0.23, 0.445)",26/52,0.5000,0.0000,69/70,0,2711.0000,3173.000,3641881,10854,0.294938
2,4B-lean gpt-4o-mini,20/70,0.2857,"(0.1932, 0.4005)",0/52,0.0000,0.0000,70/70,0,2009.0000,2531.000,1172543,4815,0.094559
3,4B-lean gemini-2.5-flash-lite,25/70,0.3571,"(0.255, 0.4741)",12/52,0.2308,0.0000,61/70,0,1758.0000,2608.000,1333491,13917,0.112508
4,4B-lean llama3.2:3b,14/70,0.2000,"(0.123, 0.3082)",4/52,0.0769,0.0000,52/70,0,3438.5000,20947.000,1184037,15835,0.000000
5,Exp3 gpt-4o-mini,26/70,0.3714,"(0.2677, 0.4885)",0/52,0.0000,0.0000,70/70,0,1426.0000,2181.000,28743,3812,0.006599
6,Exp3 gemini-lite,20/70,0.2857,"(0.1932, 0.4005)",0/52,0.0000,0.0714,70/70,0,966.0000,1977.000,32331,5427,0.005404
7,Exp3 llama3.2:3b,22/70,0.3143,"(0.2176, 0.4303)",11/52,0.2115,0.0000,70/70,0,1398.0000,2166.000,29877,4719,0.000000
8,rules 2a,22/70,0.3143,"(0.2176, 0.4303)",39/52,0.7500,0.0286,70/70,0,0.0860,0.096,0,0,0.000000
9,rules 2b,34/70,0.4857,"(0.3725, 0.6005)",15/52,0.2885,0.4857,70/70,5,1.4375,1.553,0,0,0.000000


## Extended metrics

In [4]:
ext = {k: metrics_ext.extended(res[k][1]) for k in PLAN}
pd.DataFrame([{'system': label(*k), **metrics_ext.table(ext[k])} for k in PLAN])

,system,macro_F1,wrongful_reject,over_escal_rate,missed_escal_rate,err_among_auto,cite_prec,cite_rec,mf_prec,mf_rec,cost/correct
0,4B-full gpt-4o-mini,0.4333,1/18,0.0,1.0,0.6857,0.2619,0.0464,0.009,0.062,0.012305
1,4B-full gemini-2.5-flash-lite,0.3515,0/18,0.0,1.0,0.6714,0.1273,0.2026,0.016,0.062,0.012823
2,4B-lean gpt-4o-mini,0.3600,2/18,0.0,1.0,0.7143,0.3400,0.0250,0.019,0.125,0.004728
3,4B-lean gemini-2.5-flash-lite,0.4264,0/18,0.0,1.0,0.6429,0.1664,0.1360,0.020,0.125,0.004500
4,4B-lean llama3.2:3b,0.2679,13/18,0.0,1.0,0.8000,0.0550,0.0221,0.000,0.000,0.000000


In [5]:
for k in PLAN:
    print(label(*k)); print(pd.DataFrame(ext[k]['per_class']).T.round(3).to_string()); print('decisions:', res[k][0]['decision_counts'], '\n')

4B-full gpt-4o-mini
                     support  precision  recall     f1
APPROVE                 18.0        NaN   0.000    NaN
REJECT                  19.0      0.857   0.316  0.462
REQUEST_INFORMATION     16.0      0.254   1.000  0.405
ESCALATE                17.0        NaN   0.000    NaN
decisions: {'REQUEST_INFORMATION': 63, 'REJECT': 7} 

4B-full gemini-2.5-flash-lite
                     support  precision  recall     f1
APPROVE                 18.0      0.333   0.722  0.456
REJECT                  19.0      1.000   0.158  0.273
REQUEST_INFORMATION     16.0      0.259   0.438  0.326
ESCALATE                17.0        NaN   0.000    NaN
decisions: {'REQUEST_INFORMATION': 27, 'APPROVE': 39, 'REJECT': 3, None: 1} 

4B-lean gpt-4o-mini
                     support  precision  recall    f1
APPROVE                 18.0        NaN   0.000   NaN
REJECT                  19.0      0.667   0.211  0.32
REQUEST_INFORMATION     16.0      0.250   1.000  0.40
ESCALATE                17.0    

## Grounding and policy-version metrics
Now that the policy is available, citations can be checked against the case's gold controlling clauses: **citation precision** (cited clauses that are gold), **citation recall** (gold clauses cited), nonexistent-id rate, and wrong-policy-version cases (a global policy clause of the wrong year cited for the transaction year).

In [6]:
gtb = evaluate.load_gt()
def g(k):
    recs = res[k][1]; n = len(recs)
    return {'system': label(*k), 'cites any clause': f"{sum(r['cited_any_policy'] for r in recs)}/{n}", 'nonexistent id': f"{sum(r['cited_nonexistent_clause'] for r in recs)}/{n}", 'wrong-version cases': res[k][0]['wrong_policy_version_cases'],
            'citation precision': ext[k]['citation_precision'], 'citation recall': round(ext[k]['citation_recall'], 3), 'cites >=1 gold clause': f"{sum(bool(set(r['policy_evidence']) & set(gtb[r['case_id']]['required_policy_ids'])) for r in recs)}/{n}"}
grd = pd.DataFrame([g(k) for k in PLAN]); grd

,system,cites any clause,nonexistent id,wrong-version cases,citation precision,citation recall,cites >=1 gold clause
0,4B-full gpt-4o-mini,7/70,1/70,2,0.2619,0.046,4/70
1,4B-full gemini-2.5-flash-lite,56/70,29/70,3,0.1273,0.203,26/70
2,4B-lean gpt-4o-mini,5/70,0/70,0,0.3400,0.025,3/70
3,4B-lean gemini-2.5-flash-lite,34/70,23/70,2,0.1664,0.136,15/70
4,4B-lean llama3.2:3b,6/70,1/70,4,0.0550,0.022,3/70


## Where the models succeed and fail

In [7]:
def fam(k): d = pd.DataFrame(res[k][2]); return d.groupby('case_family').correct.agg(['sum','count']).rename(columns={'sum': label(*k).replace('4B-','')[:18] + ' ok', 'count': 'n'})
F = fam(PLAN[0])
for k in PLAN[1:]: F = F.join(fam(k).drop(columns='n'))
F['group'] = pd.DataFrame(res[PLAN[0]][2]).groupby('case_family').architecture_group.first(); print(F.sort_values('n', ascending=False).to_string())
print(pd.concat({label(*k)[3:22]: pd.DataFrame(res[k][2]).groupby('architecture_group').correct.agg(['sum','count']) for k in PLAN}, axis=1))

                              full gpt-4o-mini ok  n  full gemini-2.5-fl ok  lean gpt-4o-mini ok  lean gemini-2.5-fl ok  lean llama3.2:3b ok             group
case_family                                                                                                                                                   
HOTEL_CEILING                                   2  7                      1                    2                      2                    1        B_WORKFLOW
DYNAMIC_HOTEL_DISCOVERY                         1  5                      0                    1                      1                    0   C_AGENT_DYNAMIC
APPROVAL_TIERS                                  1  5                      0                    1                      0                    0        B_WORKFLOW
DYNAMIC_DELEGATION_CHAIN                        1  5                      2                    1                      1                    0   C_AGENT_DYNAMIC
TRAINING_CAP                                  

## Plots

In [8]:
D = ['APPROVE','REJECT','REQUEST_INFORMATION','ESCALATE']
resm = {k[1] if k[0]=='full' else k[1]: res[k] for k in PLAN}   # placeholder for llm_exp.plot (per variant)
for v in ('full', 'lean'):
    llm_exp.plot('EXP04B_' + v.upper(), 'DEVELOPMENT', {m: res[(v, m)] for vv, m in PLAN if vv == v}, f'V2 Exp 4B: {v} corpus in context (70 dev claims)')
fig, ax = plt.subplots(1, 4, figsize=(24, 5))
names = [label(*k)[3:] for k in PLAN] + ['Exp3 gpt-4o-mini', 'rules 2b', 'rules 2c']; vals = [res[k][0]['correct_disposition_rate'] for k in PLAN] + [e3['gpt-4o-mini']['correct_disposition_rate'], base['rules 2b']['correct_disposition_rate'], base['rules 2c']['correct_disposition_rate']]
ax[0].barh(names, vals, color=['C0']*5 + ['C7']*3); ax[0].set_xlim(0, 1); ax[0].axvline(0.267, ls='--', c='k', lw=.8); ax[0].set_title('Correct Disposition Rate (dashed = majority class)'); ax[0].tick_params(labelsize=8)
ax[1].scatter([res[k][0]['input_tokens'] / 70 for k in PLAN], [res[k][0]['correct_disposition_rate'] for k in PLAN], s=80)
for k in PLAN: ax[1].annotate(label(*k)[3:], (res[k][0]['input_tokens'] / 70, res[k][0]['correct_disposition_rate']), fontsize=7)
ax[1].set_xlabel('input tokens per claim'); ax[1].set_ylabel('CDR'); ax[1].set_title('Accuracy vs context size')
ax[2].bar(range(len(PLAN)), [res[k][0]['total_cost_usd'] for k in PLAN], color='C2'); ax[2].set_xticks(range(len(PLAN))); ax[2].set_xticklabels([label(*k)[3:] for k in PLAN], rotation=35, fontsize=7); ax[2].set_title('Total cost, 70 dev claims (USD)')
ax[3].bar(range(len(PLAN)), [res[k][0]['median_latency_ms'] / 1000 for k in PLAN], color='C1'); ax[3].set_xticks(range(len(PLAN))); ax[3].set_xticklabels([label(*k)[3:] for k in PLAN], rotation=35, fontsize=7); ax[3].set_title('Median latency per claim (s)')
plt.tight_layout(); plt.savefig(C.RESULTS/'plots'/'exp04b_long_context_summary.png', dpi=130); plt.show()
fig, ax = plt.subplots(1, len(PLAN), figsize=(5 * len(PLAN), 4.2))
for a, k in zip(ax, PLAN):
    mm = pd.crosstab(pd.Categorical([r['expected_decision'] for r in res[k][2]], D), pd.Categorical([r['predicted_decision'] for r in res[k][2]], D), dropna=False).values
    a.imshow(mm, cmap='Blues'); a.set_xticks(range(4)); a.set_yticks(range(4)); a.set_xticklabels([d[:6] for d in D], fontsize=7); a.set_yticklabels([d[:6] for d in D], fontsize=7); a.set_title(label(*k)[3:], fontsize=9)
    for i in range(4):
        for j in range(4): a.text(j, i, mm[i, j], ha='center', va='center', color='w' if mm[i, j] > mm.max() / 2 else 'k')
plt.tight_layout(); plt.savefig(C.RESULTS/'plots'/'exp04b_confusions.png', dpi=120); plt.show()

/var/folders/3c/pd4v_b0x2qx_d95rzy8l4w640000gn/T/ipykernel_51343/1307821733.py:13: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(C.RESULTS/'plots'/'exp04b_long_context_summary.png', dpi=130); plt.show()
/var/folders/3c/pd4v_b0x2qx_d95rzy8l4w640000gn/T/ipykernel_51343/1307821733.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.savefig(C.RESULTS/'plots'/'exp04b_confusions.png', dpi=120); plt.show()


## Failure table (first 30 wrong decisions of the best long-context system)

In [9]:
best = max(PLAN, key=lambda k: res[k][0]['correct']); print('best system:', label(*best), res[best][0]['correct'], '/ 70')
notes = {c['case_id']: c['employee_description'][:100] for c in cases}; rows = []
for r, e in zip(res[best][1], res[best][2]):
    if not e['correct']: rows.append({'case_id': r['case_id'], 'family': e['case_family'], 'expected': e['expected_decision'], 'predicted': r['predicted_decision'], 'cited': ', '.join(r['policy_evidence'][:3]), 'why': (r['explanation'] or '')[:120]})
pd.DataFrame(rows).head(30)

best system: 4B-lean gemini-2.5-flash-lite 25 / 70


,case_id,family,expected,predicted,cited,why
0,X2-003,FX_THRESHOLD,APPROVE,REQUEST_INFORMATION,,"The claim is for a software subscription. The business purpose, software owner, and manager approval are m..."
1,X2-006,DUPLICATE_CHECK,REJECT,APPROVE,"GEP24-1.3, GEP24-2.1, SWE-1.1","The claim is approved as it meets the documentation requirements, was submitted within the 60-day window, ..."
2,X2-010,CONFERENCE_FEE,REJECT,REQUEST_INFORMATION,"TRN-1.1, TRN-2.4","The claim is for a conference registration fee, which requires a learning plan identifier and an approved ..."
3,X2-011,HOTEL_CEILING,APPROVE,REQUEST_INFORMATION,,"The claim is for hotel accommodation, which requires an approved business trip and the employee's grade to..."
4,X2-012,HOTEL_EXCEPTION,REJECT,REQUEST_INFORMATION,"GEP25-1.3, D04, TRV-3.2",The claim requires additional information to determine compliance with hotel accommodation policies. Speci...
5,X2-015,APPROVAL_TIERS,ESCALATE,REQUEST_INFORMATION,,"The claim is for a meal with external guests, but the number of external attendees and the business purpos..."
6,X2-018,DYNAMIC_DELEGATION_CHAIN,REQUEST_INFORMATION,None,,
7,X2-024,AIRFARE_CABIN,REQUEST_INFORMATION,APPROVE,"GEP24-1.3, GEP24-2.1, AIR-1.1",The claim is approved as the economy class airfare is within policy. The flight duration meets the criteri...
8,X2-028,AIRFARE_CABIN,REJECT,None,,
9,X2-029,HOTEL_CEILING,APPROVE,REQUEST_INFORMATION,,"The claim requires additional information to determine compliance with hotel accommodation policies, speci..."


## Decision gate for RAG
RAG must later show at least one advantage over these long-context baselines. The table lists the long-context reference values for each gate criterion.

In [10]:
gate = pd.DataFrame([{'system': label(*k)[3:], 'CDR': res[k][0]['correct_disposition_rate'], 'false-approval rate': res[k][0]['false_approval_rate'], 'wrong-version cases': res[k][0]['wrong_policy_version_cases'], 'citation precision': ext[k]['citation_precision'],
                      'input tokens/claim': round(res[k][0]['input_tokens'] / 70), 'cost/claim $': round(res[k][0]['total_cost_usd'] / 70, 5), 'median latency s': round(res[k][0]['median_latency_ms'] / 1000, 2)} for k in PLAN]); gate

,system,CDR,false-approval rate,wrong-version cases,citation precision,input tokens/claim,cost/claim $,median latency s
0,full gpt-4o-mini,0.3143,0.0000,2,0.2619,49254,0.00387,2.83
1,full gemini-2.5-flash-lite,0.3286,0.5000,3,0.1273,52027,0.00421,2.71
2,lean gpt-4o-mini,0.2857,0.0000,0,0.3400,16751,0.00135,2.01
3,lean gemini-2.5-flash-lite,0.3571,0.2308,2,0.1664,19050,0.00161,1.76
4,lean llama3.2:3b,0.2000,0.0769,4,0.0550,16915,0.00000,3.44


## What was done and what it means
- **Result (70 dev claims, corpus in the prompt, no enterprise records):** full corpus: gpt-4o-mini 22/70 = 31% (CI 22-43%), gemini-2.5-flash-lite 23/70 = 33% (23-45%). Lean corpus: gpt-4o-mini 20/70 = 29%, gemini 25/70 = 36% (26-47%), llama3.2:3b 14/70 = 20% (12-31%). **All are at chance level (majority class 26.7%) and no better than Exp 3 without any policy** (gpt-4o-mini 37%, gemini 29%, llama 31%), and far below the rules (2b 49%, 2c 69%). Giving the model the whole policy did not help.
- **Why: the policy is not the missing ingredient.** 110 of the 150 claims (groups B and C) depend on enterprise records (approvals, travel requests, exceptions, delegations, previous expenses) that are not in the claim. Even the self-contained group A, which the policy alone should settle, reaches only 7-10 of 18 (best: gemini lean 10/18 = 56%). The models answer REQUEST_INFORMATION for most claims (63/70 for gpt-4o-mini) and never ESCALATE (missed-escalation rate 100% for every system): they can see that "something is missing" but cannot tell what an approval record would say.
- **Full vs lean:** removing the interpretive commentary (76% of the corpus) changed accuracy by only 2 and 2 claims in opposite directions (gpt-4o-mini 22 -> 20, gemini 23 -> 25): not distinguishable, but lean uses a third of the tokens and cost.
- **Safety and traceability:** gemini approves aggressively with the full corpus (39 APPROVE decisions, **26/52 = 50% false approvals**) and 12/52 with lean; gpt-4o-mini and the local model stay low only because they almost never approve. Citations are poor even with the corpus available: gpt-4o-mini cites a clause id for only 5-7 of 70 claims, gemini cites for 34-56 but **29/70 (full) and 23/70 (lean) cite an id that does not exist** (for example document ids like D04), with citation precision 0.13-0.17. Wrong-policy-version citations occur in 0-4 claims. Llama returned invalid JSON for 18 of 70 claims (schema-valid 52/70) and asks for a JSON answer badly at 17k tokens.
- **Cost and latency:** input is 16.8k (lean) to 49-52k (full) tokens per claim. The dev runs cost $0.27 / $0.29 (full, gpt / gemini) and $0.09 / $0.11 (lean), $0.77 in total against a projection of $1.21; the paid runs were cheaper than projected, probably because of provider-side prompt caching of the shared corpus prefix (not verified). Median latency 1.8-2.8 s (paid), 3.4 s (llama, p95 21 s). Cost per correct decision is about $0.005 (lean) to $0.013 (full).
- **Decision gate:** long context is **not a competitive accuracy baseline** here, so RAG cannot be judged "unnecessary" on accuracy grounds. RAG must still show a gain on cost or latency (fewer tokens per claim than 17-52k), false approvals, version precision or traceability (correct, existing clause ids). Because the accuracy ceiling comes from missing records rather than missing policy, the experiments that supply evidence (12 rules + RAG, 16 evidence oracle, 18 workflow) are where accuracy can actually move.
- **Caveats:** dev split only, 70 claims, one prompt, temperature 0. Total spend now $1.35 of the $3.50 cap. Final test untouched.
- **Decision:** carry forward the lean corpus as the cheaper long-context reference. Next: Exp 5 naive RAG.